# Vault AWS Secrets Engine 통합 가이드

이 노트북은 AWS Secrets Engine의 두 가지 주요 인증 방식을 모두 다룹니다.

1. **AccessKey 방식**: IAM User의 고정 키를 Vault에 등록 (로컬/테스트 환경 권장)
2. **IAM Role (IRSA) 방식**: Vault Pod의 identity를 사용하여 인증 (EKS/운영 환경 권장)

HashiCorp 공식 문서 - https://developer.hashicorp.com/vault/docs/secrets/aws

## 0. 환경변수 설정

In [1]:
# direnv 환경변수 로드
direnv allow
eval $(direnv export bash)

# 1. AccessKey 방식 설정
export KEY_MOUNT_PATH="aws-accesskey"

# 2. IAM Role (IRSA) 방식 설정
export ROLE_MOUNT_PATH="aws-irsa"

echo "VAULT_ADDR      : $VAULT_ADDR"
echo "AWS_REGION      : $AWS_REGION"
echo "KEY_MOUNT_PATH  : $KEY_MOUNT_PATH"
echo "ROLE_MOUNT_PATH : $ROLE_MOUNT_PATH"

direnv: loading ~/workspace/vault-aws/vault-example/.envrc
direnv: export +AWS_ACCESS_KEY_ID +AWS_REGION +AWS_SECRET_ACCESS_KEY +VAULT_ADDR +VAULT_ASSUME_TARGET_ROLE_ARN +VAULT_ROLE_ARN +VAULT_TOKEN
VAULT_ADDR      : http://k8s-vault-vaultui-328e838693-e9ce0d71f5f8868f.elb.ap-northeast-2.amazonaws.com:8200
AWS_REGION      : ap-northeast-2
KEY_MOUNT_PATH  : aws-accesskey
ROLE_MOUNT_PATH : aws-irsa



## 1. AWS Secrets Engine 마운트

In [2]:
# [방법 1] AccessKey 방식용 마운트
vault secrets enable \
    -path="$KEY_MOUNT_PATH" \
    -default-lease-ttl=5m \
    -max-lease-ttl=1h \
    aws

Success! Enabled the aws secrets engine at: aws-accesskey/



In [3]:
# [방법 2] IAM Role (IRSA) 방식용 마운트
vault secrets enable \
    -path="$ROLE_MOUNT_PATH" \
    -default-lease-ttl=5m \
    -max-lease-ttl=1h \
    aws

Success! Enabled the aws secrets engine at: aws-irsa/



## 2. Root Config 설정

In [4]:
# [방법 1] AccessKey/SecretKey 직접 등록
vault write "$KEY_MOUNT_PATH/config/root" \
  access_key="$AWS_ACCESS_KEY_ID" \
  secret_key="$AWS_SECRET_ACCESS_KEY" \
  region="$AWS_REGION"

Success! Data written to: aws-accesskey/config/root



In [5]:
# [방법 2] IRSA 기반 설정 (region만 지정)
vault write "$ROLE_MOUNT_PATH/config/root" \
  region="$AWS_REGION"

Success! Data written to: aws-irsa/config/root



## 3. Vault Role 생성

Vault Role은 **어떤 AWS 권한으로** 동적 자격증명을 발급할지 정의합니다.

| `credential_type` | 설명 |
|---|---|
| `iam_user` | IAM User + AccessKey를 동적으로 생성 |
| `assumed_role` | STS AssumeRole로 임시 자격증명 발급 |
| `federation_token` | STS GetFederationToken으로 임시 자격증명 발급 |

In [8]:
# [방법 1용] AccessKey 경로 Roles
# 1. IAM User 방식 Role
vault write "$KEY_MOUNT_PATH/roles/key-iam-user-role" \
    credential_type=iam_user \
    policy_arns="arn:aws:iam::aws:policy/SecretsManagerReadWrite"

# 2. STS AssumeRole 방식 Role
vault write "$KEY_MOUNT_PATH/roles/key-sts-role" \
    credential_type=assumed_role \
    role_arns="$VAULT_ASSUME_TARGET_ROLE_ARN"

Success! Data written to: aws-accesskey/roles/key-iam-user-role
Success! Data written to: aws-accesskey/roles/key-sts-role



In [9]:
# [방법 2용] IRSA 경로 Roles
# 1. IAM User 방식 Role
vault write "$ROLE_MOUNT_PATH/roles/role-iam-user-role" \
    credential_type=iam_user \
    policy_arns="arn:aws:iam::aws:policy/SecretsManagerReadWrite"

# 2. STS AssumeRole 방식 Role
vault write "$ROLE_MOUNT_PATH/roles/role-sts-role" \
    credential_type=assumed_role \
    role_arns="$VAULT_ASSUME_TARGET_ROLE_ARN"

Success! Data written to: aws-irsa/roles/role-iam-user-role
Success! Data written to: aws-irsa/roles/role-sts-role



## 4. 동적 자격증명 발급 및 테스트

In [10]:
# [방법 1 테스트 데이터] AccessKey 경로에서 발급
echo "=== AccessKey 기반 발급 ==="
IAM_CREDS=$(vault read -format=json "$KEY_MOUNT_PATH/creds/key-iam-user-role")
export KEY_IAM_LEASE_ID=$(echo $IAM_CREDS | python3 -c "import sys,json; d=json.load(sys.stdin); print(d['lease_id'])")

STS_CREDS=$(vault read -format=json "$KEY_MOUNT_PATH/sts/key-sts-role")
export KEY_STS_LEASE_ID=$(echo $STS_CREDS | python3 -c "import sys,json; d=json.load(sys.stdin); print(d['lease_id'])")

# 테스트를 위한 공통 변수 할당 (STS 기준)
export DYNA_ACCESS_KEY=$(echo $STS_CREDS | python3 -c "import sys,json; d=json.load(sys.stdin); print(d['data']['access_key'])")
export DYNA_SECRET_KEY=$(echo $STS_CREDS | python3 -c "import sys,json; d=json.load(sys.stdin); print(d['data']['secret_key'])")
export DYNA_SESSION_TOKEN=$(echo $STS_CREDS | python3 -c "import sys,json; d=json.load(sys.stdin); print(d['data'].get('security_token', ''))")

echo "AccessKey 기반 Lease 발급 완료."

=== AccessKey 기반 발급 ===
Error reading aws-accesskey/sts/key-sts-role: Error making API request.

URL: GET http://k8s-vault-vaultui-328e838693-e9ce0d71f5f8868f.elb.ap-northeast-2.amazonaws.com:8200/v1/aws-accesskey/sts/key-sts-role
Code: 400. Errors:

* Error assuming role: AccessDenied: User: arn:aws:iam::341689148868:user/vault-local-credential-05ff97 is not authorized to perform: sts:AssumeRole on resource: arn:aws:iam::341689148868:role/vault-role-sts-target
	status code: 403, request id: e174a373-83f6-42b7-b25f-22ad7cc22966
Traceback (most recent call last):
  File "<string>", line 1, in <module>
  File "/Library/Developer/CommandLineTools/Library/Frameworks/Python3.framework/Versions/3.9/lib/python3.9/json/__init__.py", line 293, in load
    return loads(fp.read(),
  File "/Library/Developer/CommandLineTools/Library/Frameworks/Python3.framework/Versions/3.9/lib/python3.9/json/__init__.py", line 346, in loads
    return _default_decoder.decode(s)
  File "/Library/Developer/CommandL

In [ ]:
# [방법 2 테스트 데이터] IRSA 경로에서 발급
echo "=== IRSA 기반 발급 ==="
IAM_CREDS=$(vault read -format=json "$ROLE_MOUNT_PATH/creds/role-iam-user-role")
export ROLE_IAM_LEASE_ID=$(echo $IAM_CREDS | python3 -c "import sys,json; d=json.load(sys.stdin); print(d['lease_id'])")

STS_CREDS=$(vault read -format=json "$ROLE_MOUNT_PATH/sts/role-sts-role")
export ROLE_STS_LEASE_ID=$(echo $STS_CREDS | python3 -c "import sys,json; d=json.load(sys.stdin); print(d['lease_id'])")

# 테스트를 위한 공통 변수 할당 (STS 기준)
export DYNA_ACCESS_KEY=$(echo $STS_CREDS | python3 -c "import sys,json; d=json.load(sys.stdin); print(d['data']['access_key'])")
export DYNA_SECRET_KEY=$(echo $STS_CREDS | python3 -c "import sys,json; d=json.load(sys.stdin); print(d['data']['secret_key'])")
export DYNA_SESSION_TOKEN=$(echo $STS_CREDS | python3 -c "import sys,json; d=json.load(sys.stdin); print(d['data'].get('security_token', ''))")

echo "IRSA 기반 Lease 발급 완료."

In [ ]:
# [공통] AWS CLI 테스트
export AWS_ACCESS_KEY_ID="$DYNA_ACCESS_KEY"
export AWS_SECRET_ACCESS_KEY="$DYNA_SECRET_KEY"
export AWS_SESSION_TOKEN="$DYNA_SESSION_TOKEN"
export AWS_DEFAULT_REGION="$AWS_REGION"

echo "=== 1. identity 확인 ==="
aws sts get-caller-identity

echo ""
echo "=== 2. S3 목록 조회 ==="
aws s3 ls

## 5. Lease 관리 (목록 / 갱신 / 취소)

In [ ]:
# 모든 활성 리스 목록 확인
echo "=== [AccessKey] IAM User Leases ==="
vault list "sys/leases/lookup/$KEY_MOUNT_PATH/creds/key-iam-user-role/" || echo "(없음)"

echo ""
echo "=== [IRSA] IAM User Leases ==="
vault list "sys/leases/lookup/$ROLE_MOUNT_PATH/creds/role-iam-user-role/" || echo "(없음)"

In [ ]:
# 리스 갱신 테스트 (IAM User만 가능)
echo "=== IAM User Lease 갱신 시도 ==="
vault lease renew -increment=3600 "$KEY_IAM_LEASE_ID"

In [ ]:
# 특정 리스 개별 취소
vault lease revoke "$KEY_IAM_LEASE_ID"
vault lease revoke "$ROLE_IAM_LEASE_ID"

## 6. (정리) 리소스 삭제 및 언마운트

In [ ]:
# AccessKey 기반 엔진 정리
vault secrets disable "$KEY_MOUNT_PATH"

Success! Disabled the secrets engine (if it existed) at: aws-accesskey/
✅ AccessKey 기반 엔진 언마운트 완료



In [ ]:
# IRSA 기반 엔진 정리
vault secrets disable "$ROLE_MOUNT_PATH"

Success! Disabled the secrets engine (if it existed) at: aws-irsa/
✅ IRSA 기반 엔진 언마운트 완료

